# 02 Análise exploratória (EDA)

Objetivo: conhecer os dados antes dos testes. Quantas pessoas responderam cada área, quanto falta de dado, como os grupos se distribuem e as primeiras proporções com peso. Nada aqui é resultado final: os números com margem de erro saem no notebook 03.

In [1]:
import os
import numpy as np
import pandas as pd

def encontrar_raiz_projeto(pasta_atual):
    # Sobe de pasta em pasta até achar a que tem src/harmonizacao.py
    pasta = os.path.abspath(pasta_atual)
    while True:
        if os.path.exists(os.path.join(pasta, 'src', 'harmonizacao.py')):
            return pasta
        pasta_pai = os.path.dirname(pasta)
        if pasta_pai == pasta:
            raise FileNotFoundError('Não achei src/harmonizacao.py subindo a partir de ' + pasta_atual)
        pasta = pasta_pai


raiz_projeto = encontrar_raiz_projeto(os.getcwd())

# Mostra todas as colunas nas tabelas
pd.set_option('display.max_columns', None)

# Arquivo principal: uma linha por pessoa, com os dados da casa dela
dados = pd.read_csv(os.path.join(raiz_projeto, 'data', 'processed', 'dados_harmonizados.csv'))

print(dados.shape)
dados.head() 

(66976, 32)


,chave_domicilio,ID_MORADOR,ano,idade,sexo,raca,escolaridade,g1_agreg,transacoes_financeiras,peso_amostral_individuo,G1_A,G1_B,G1_C,G1_D,G1_E,G1_F,G1_G,G2_A,G2_B,G2_C,G2_D,G2_E,G2_F,G2_G,G1_H,G2_H,renda_familiar,classe_economica,regiao,area_urbano_rural,velocidade_conexao,valor_conexao
0,2,1,2023,36,2,3.0,2,1,0.0,344.582890,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,4,4,1,NaN,9.0
1,3,2,2023,53,2,1.0,4,1,0.0,1765.454757,0.0,1.0,0.0,1.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,4.0,2,4,1,NaN,9.0
2,4,1,2023,71,2,2.0,2,0,0.0,474.963986,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4,4,1,baixa,7.0
3,5,3,2023,37,2,1.0,3,0,0.0,599.592121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3,4,1,NaN,7.0
4,6,1,2023,30,1,3.0,3,0,1.0,416.368061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4,4,1,media,8.0


In [3]:
domicilios = pd.read_csv(os.path.join(raiz_projeto, 'data', 'processed', 'domicilios_harmonizado.csv'))

print(domicilios.shape)
domicilios.head()

(75008, 9)


,chave_domicilio,ano,peso_amostral_domicilio,renda_familiar,classe_economica,regiao,area_urbano_rural,velocidade_conexao,valor_conexao
0,2,2023,320.771158,1.0,4,4,1,NaN,9.0
1,3,2023,949.725387,4.0,2,4,1,NaN,9.0
2,4,2023,320.771158,2.0,4,4,1,baixa,7.0
3,5,2023,602.051569,2.0,3,4,1,NaN,7.0
4,6,2023,320.771158,2.0,4,4,1,media,8.0


In [4]:
# Saúde (área B): quantos resolveram, quantos não, quantos não foram perguntados, por ano
pd.crosstab(dados['ano'], dados['G2_B'].fillna('não perguntado'))

G2_B,0.0,1.0,não perguntado
ano,,,
2023,4026,927,16318
2024,3050,1005,17115
2025,4773,1204,18558


In [5]:
pd.crosstab(dados['ano'], dados['G2_E'].fillna('não perguntado'))

G2_E,0.0,1.0,não perguntado
ano,,,
2023,1862,1714,17695
2024,1419,1528,18223
2025,1886,1989,20660


In [25]:
def porcentagem_com_peso(resposta, peso):
    # % de 1 numa coluna 0/1, usando o peso de cada pessoa e ignorando os vazios
    tem_resposta = resposta.notna()
    y = resposta[tem_resposta]
    w = peso[tem_resposta]
    return 100 * (w * y).sum() / w.sum()

def n_efetivo(peso):
    return peso.sum() ** 2 / (peso ** 2).sum()

nomes_areas = {'A': 'documentos', 'B': 'saúde', 'C': 'educação', 'D': 'trabalho',
               'E': 'impostos', 'F': 'polícia', 'G': 'transporte', 'H': 'justiça'}

linhas = []

for ano in [2023, 2024, 2025]:
    d = dados[dados['ano'] == ano]

    for letra, nome in nomes_areas.items():
        if letra == 'H' and ano != 2025:
            continue

        g1 = d[f'G1_{letra}']
        g2 = d[f'G2_{letra}']

        linhas.append({
            'ano': ano,
            'area': nome,
            'responderam_G1': g1.notna().sum(),
            'tentaram': (g1 == 1).sum(),
            'responderam_G2': g2.notna().sum(),
            'resolveram': (g2 == 1).sum(),
            'pct_com_peso': round(porcentagem_com_peso(g2, d['peso_amostral_individuo']), 1)
        })

contagem = pd.DataFrame(linhas)
contagem

,ano,area,responderam_G1,tentaram,responderam_G2,resolveram,pct_com_peso
0,2023,documentos,14957,3859,3840,1082,34.9
1,2023,saúde,14959,4982,4953,927,21.7
2,2023,educação,14953,3025,3017,972,34.0
3,2023,trabalho,14954,4167,4147,1209,34.0
4,2023,impostos,14947,3591,3576,1714,55.1
5,2023,polícia,14959,1468,1459,588,45.0
6,2023,transporte,14958,1657,1639,520,42.5
7,2024,documentos,15623,3482,3466,1084,30.9
8,2024,saúde,15625,4071,4055,1005,25.6
9,2024,educação,15635,2445,2439,879,36.1


In [14]:
contagem['pct_resolveu'] = (100 * contagem['resolveram'] / contagem['responderam_G2']).round(1)
contagem['diferenca'] = round(contagem['pct_com_peso'] - contagem['pct_resolveu'], 1)

contagem

,ano,area,responderam_G1,tentaram,responderam_G2,resolveram,pct_com_peso,pct_resolveu,diferenca
0,2023,documentos,14957,3859,3840,1082,34.9,28.2,6.7
1,2023,saúde,14959,4982,4953,927,21.7,18.7,3.0
2,2023,educação,14953,3025,3017,972,34.0,32.2,1.8
3,2023,trabalho,14954,4167,4147,1209,34.0,29.2,4.8
4,2023,impostos,14947,3591,3576,1714,55.1,47.9,7.2
5,2023,polícia,14959,1468,1459,588,45.0,40.3,4.7
6,2023,transporte,14958,1657,1639,520,42.5,31.7,10.8
7,2024,documentos,15623,3482,3466,1084,30.9,31.3,-0.4
8,2024,saúde,15625,4071,4055,1005,25.6,24.8,0.8
9,2024,educação,15635,2445,2439,879,36.1,36.0,0.1


In [15]:
# Passo 4: saúde (G2_B) em 2024, separando por faixa de renda

# Traduz o código da renda pra um nome (igual ao nomes_areas, mas a chave é um número)
nomes_renda = {1: 'até 1 SM', 2: '1 a 2 SM', 3: '2 a 3 SM', 4: '3 a 5 SM',
               5: '5 a 10 SM', 6: 'mais de 10 SM', 7: 'não tem renda'}

d24 = dados[dados['ano'] == 2024]          # só 2024
saude_2024 = d24[d24['G2_B'].notna()]      # dessas pessoas, só quem respondeu a pergunta de saúde

# Conta quantas pessoas tem em cada código de renda
saude_2024['renda_familiar'].value_counts(dropna=False).sort_index()

renda_familiar
1.0    1224
2.0    1131
3.0     761
4.0     445
5.0     205
6.0      57
7.0      53
NaN     179
Name: count, dtype: int64

In [16]:
linhas = []   

for codigo, nome in nomes_renda.items():
    grupo = saude_2024[saude_2024['renda_familiar'] == codigo]  
    n = len(grupo)                                               
    resolveram = (grupo['G2_B'] == 1).sum()                     
    linhas.append({
        'renda': nome,
        'n': n,
        'resolveram': resolveram,
        'pct_sem_peso': round(100 * resolveram / n, 1),
        'pct_com_peso': round(porcentagem_com_peso(grupo['G2_B'], grupo['peso_amostral_individuo']), 1),
    })

saude_renda = pd.DataFrame(linhas)
saude_renda

,renda,n,resolveram,pct_sem_peso,pct_com_peso
0,até 1 SM,1224,238,19.4,23.4
1,1 a 2 SM,1131,276,24.4,18.5
2,2 a 3 SM,761,204,26.8,33.6
3,3 a 5 SM,445,148,33.3,24.7
4,5 a 10 SM,205,59,28.8,34.1
5,mais de 10 SM,57,19,33.3,38.8
6,não tem renda,53,11,20.8,6.3


In [17]:
impostos_2024 = d24[d24['G2_E'].notna()]

linhas = []
for codigo, nome in nomes_renda.items():
    grupo = impostos_2024[impostos_2024['renda_familiar'] == codigo]
    n = len(grupo)
    resolveram = (grupo['G2_E'] == 1).sum()
    linhas.append({
        'renda': nome,
        'n': n,
        'resolveram': resolveram,
        'pct_sem_peso': round(100 * resolveram / n, 1),
        'pct_com_peso': round(porcentagem_com_peso(grupo['G2_E'], grupo['peso_amostral_individuo']), 1),
    })

impostos_renda = pd.DataFrame(linhas)
impostos_renda

,renda,n,resolveram,pct_sem_peso,pct_com_peso
0,até 1 SM,407,128,31.4,37.8
1,1 a 2 SM,656,285,43.4,47.5
2,2 a 3 SM,646,345,53.4,53.1
3,3 a 5 SM,562,339,60.3,60.6
4,5 a 10 SM,365,237,64.9,66.7
5,mais de 10 SM,116,85,73.3,81.4
6,não tem renda,15,8,53.3,57.6


In [18]:
saude_renda

,renda,n,resolveram,pct_sem_peso,pct_com_peso
0,até 1 SM,1224,238,19.4,23.4
1,1 a 2 SM,1131,276,24.4,18.5
2,2 a 3 SM,761,204,26.8,33.6
3,3 a 5 SM,445,148,33.3,24.7
4,5 a 10 SM,205,59,28.8,34.1
5,mais de 10 SM,57,19,33.3,38.8
6,não tem renda,53,11,20.8,6.3


In [19]:
grupo = saude_2024[saude_2024['renda_familiar'] == 7]          # só "não tem renda"
mais_pesados = grupo.sort_values('peso_amostral_individuo', ascending=False)
mais_pesados[['G2_B', 'peso_amostral_individuo']].head(5)      # as 5 pessoas mais pesadas

,G2_B,peso_amostral_individuo
36163,0.0,264638.091466
36737,0.0,41228.456476
36424,0.0,26283.404828
32766,0.0,16887.965716
36842,0.0,14152.290259


In [ ]:
pesos = grupo['peso_amostral_individuo']
round(100 * pesos.max() / pesos.sum(), 1)

In [26]:
# Passo 1 de 4: uma função que monta a tabela de qualquer área, para qualquer grupo
def tabela_por_grupo(base, coluna_area, coluna_grupo, nomes):
    # base:          os dados de um ano (ex.: d24)
    # coluna_area:   a pergunta G2 da área (ex.: 'G2_B')
    # coluna_grupo:  a coluna que separa os grupos (ex.: 'renda_familiar')
    # nomes:         o dicionário que traduz os códigos do grupo (ex.: nomes_renda)

    responderam = base[base[coluna_area].notna()]      # só quem respondeu a pergunta da área

    linhas = []
    for codigo, nome in nomes.items():
        grupo = responderam[responderam[coluna_grupo] == codigo]
        n = len(grupo)
        resolveram = (grupo[coluna_area] == 1).sum()
        linhas.append({
            'categoria': nome,
            'n': n,
            'resolveram': resolveram,
            'pct_sem_peso': round(100 * resolveram / n, 1),
            'pct_com_peso': round(porcentagem_com_peso(grupo[coluna_area], grupo['peso_amostral_individuo']), 1),
            'n_efetivo': round(n_efetivo(grupo['peso_amostral_individuo'])),
        })

    return pd.DataFrame(linhas)      # devolve a tabela pronta pra quem chamou a função

In [ ]:
# Teste: tem que dar igual à sua tabela saude_renda (agora com o n_efetivo no fim)
tabela_por_grupo(d24, 'G2_B', 'renda_familiar', nomes_renda)

,categoria,n,resolveram,pct_sem_peso,pct_com_peso,n_efetivo
0,branca,1224,238,19.4,23.4,231
1,preta,1131,276,24.4,18.5,71
2,parda,761,204,26.8,33.6,40
3,amarela,445,148,33.3,24.7,42
4,indígena,205,59,28.8,34.1,25


In [31]:
nomes_raca = {1: 'branca', 2: 'preta', 3: 'parda', 4: 'amarela', 5: 'indígena'}

saude_raca = tabela_por_grupo(d24, 'G2_B', 'raca', nomes_raca) 
saude_raca

,categoria,n,resolveram,pct_sem_peso,pct_com_peso,n_efetivo
0,branca,1099,306,27.8,30.4,67
1,preta,603,134,22.2,20.9,17
2,parda,2140,509,23.8,21.5,145
3,amarela,115,35,30.4,48.0,23
4,indígena,65,13,20.0,36.2,10


In [32]:
# Passo 2 de 4 (parte A): os 4 grupos de uma vez, para saúde em 2024

nomes_escolaridade = {1: 'analfabeto/infantil', 2: 'fundamental', 3: 'médio', 4: 'superior'}
nomes_regiao = {1: 'Sudeste', 2: 'Nordeste', 3: 'Sul', 4: 'Norte', 5: 'Centro-Oeste'}

# Cada coluna de grupo aponta pro seu dicionário de nomes
grupos = {
    'renda_familiar': nomes_renda,
    'raca': nomes_raca,
    'escolaridade': nomes_escolaridade,
    'regiao': nomes_regiao,
}

tabelas = []                                   # uma lista de tabelas (antes era uma lista de linhas)
for coluna_grupo, nomes in grupos.items():
    tabela = tabela_por_grupo(d24, 'G2_B', coluna_grupo, nomes)
    tabela['dimensao'] = coluna_grupo          # cria uma coluna dizendo de qual grupo veio cada linha
    tabelas.append(tabela)

saude_2024_todos = pd.concat(tabelas, ignore_index=True)   # empilha as 4 tabelas numa só
saude_2024_todos

,categoria,n,resolveram,pct_sem_peso,pct_com_peso,n_efetivo,dimensao
0,até 1 SM,1224,238,19.4,23.4,231,renda_familiar
1,1 a 2 SM,1131,276,24.4,18.5,71,renda_familiar
2,2 a 3 SM,761,204,26.8,33.6,40,renda_familiar
3,3 a 5 SM,445,148,33.3,24.7,42,renda_familiar
4,5 a 10 SM,205,59,28.8,34.1,25,renda_familiar
5,mais de 10 SM,57,19,33.3,38.8,15,renda_familiar
6,não tem renda,53,11,20.8,6.3,3,renda_familiar
7,branca,1099,306,27.8,30.4,67,raca
8,preta,603,134,22.2,20.9,17,raca
9,parda,2140,509,23.8,21.5,145,raca


In [33]:
# Passo 2 de 4 (parte A): os 4 grupos de uma vez, para saúde em 2024

nomes_escolaridade = {1: 'analfabeto/infantil', 2: 'fundamental', 3: 'médio', 4: 'superior'}
nomes_regiao = {1: 'Sudeste', 2: 'Nordeste', 3: 'Sul', 4: 'Norte', 5: 'Centro-Oeste'}

# Cada coluna de grupo aponta pro seu dicionário de nomes
grupos = {
    'renda_familiar': nomes_renda,
    'raca': nomes_raca,
    'escolaridade': nomes_escolaridade,
    'regiao': nomes_regiao,
}

tabelas = []                                   # uma lista de tabelas (antes era uma lista de linhas)
for coluna_grupo, nomes in grupos.items():
    tabela = tabela_por_grupo(d24, 'G2_E', coluna_grupo, nomes)
    tabela['dimensao'] = coluna_grupo          # cria uma coluna dizendo de qual grupo veio cada linha
    tabelas.append(tabela)

impostos_2024_todos = pd.concat(tabelas, ignore_index=True)   # empilha as 4 tabelas numa só
impostos_2024_todos

,categoria,n,resolveram,pct_sem_peso,pct_com_peso,n_efetivo,dimensao
0,até 1 SM,407,128,31.4,37.8,105,renda_familiar
1,1 a 2 SM,656,285,43.4,47.5,81,renda_familiar
2,2 a 3 SM,646,345,53.4,53.1,45,renda_familiar
3,3 a 5 SM,562,339,60.3,60.6,54,renda_familiar
4,5 a 10 SM,365,237,64.9,66.7,48,renda_familiar
5,mais de 10 SM,116,85,73.3,81.4,14,renda_familiar
6,não tem renda,15,8,53.3,57.6,5,renda_familiar
7,branca,950,568,59.8,64.4,116,raca
8,preta,393,180,45.8,32.9,11,raca
9,parda,1462,707,48.4,54.6,114,raca


In [36]:
# Qual dimensão tem a maior diferença em impostos (2024)?

# 1) Fica só com as linhas confiáveis (n efetivo de 10 pra cima)
confiaveis = impostos_2024_todos[impostos_2024_todos['n_efetivo'] >= 10]

# 2) Para cada dimensão, pega a maior e a menor % com peso
resumo = confiaveis.groupby('dimensao')['pct_com_peso'].agg(['max', 'min'])

# 3) Calcula a diferença entre elas e ordena da maior pra menor
resumo['diferenca'] = resumo['max'] - resumo['min']
resumo.sort_values('diferenca', ascending=False)

,max,min,diferenca
dimensao,,,
renda_familiar,81.4,37.8,43.6
raca,67.7,32.9,34.8
regiao,66.9,34.6,32.3
escolaridade,62.6,39.5,23.1


In [37]:
tabelas = []
for ano in [2023, 2024, 2025]:
    d = dados[dados['ano'] == ano]                       # os dados de um ano

    for letra, nome_area in nomes_areas.items():
        if letra == 'H' and ano != 2025:                 # justiça só existe em 2025
            continue

        for coluna_grupo, nomes in grupos.items():
            tabela = tabela_por_grupo(d, f'G2_{letra}', coluna_grupo, nomes)
            tabela['ano'] = ano                          # anota de onde veio cada linha
            tabela['area'] = nome_area
            tabela['dimensao'] = coluna_grupo
            tabelas.append(tabela)

tabelao = pd.concat(tabelas, ignore_index=True)

# Põe as colunas numa ordem boa de ler
tabelao = tabelao[['ano', 'area', 'dimensao', 'categoria', 'n', 'resolveram',
                   'pct_sem_peso', 'pct_com_peso', 'n_efetivo']]

print(tabelao.shape)
tabelao.head(25)

(462, 9)


,ano,area,dimensao,categoria,n,resolveram,pct_sem_peso,pct_com_peso,n_efetivo
0,2023,documentos,renda_familiar,até 1 SM,1028,209,20.3,18.9,84
1,2023,documentos,renda_familiar,1 a 2 SM,1040,273,26.2,40.6,32
2,2023,documentos,renda_familiar,2 a 3 SM,659,212,32.2,48.3,54
3,2023,documentos,renda_familiar,3 a 5 SM,498,176,35.3,29.8,44
4,2023,documentos,renda_familiar,5 a 10 SM,241,96,39.8,31.9,38
5,2023,documentos,renda_familiar,mais de 10 SM,112,46,41.1,36.6,17
6,2023,documentos,renda_familiar,não tem renda,65,10,15.4,12.7,11
7,2023,documentos,raca,branca,1031,339,32.9,37.7,57
8,2023,documentos,raca,preta,587,146,24.9,40.9,52
9,2023,documentos,raca,parda,1999,539,27.0,31.5,146


In [ ]:
# Confere o tamanho
assert tabelao.shape == (462, 9)

# Fica só com as linhas frágeis (n efetivo menor que 10)
frageis = tabelao[tabelao['n_efetivo'] < 10]
print(len(frageis), 'de', len(tabelao), 'linhas são frágeis')

# Quantas vezes cada grupo aparece como frágil, somando anos e áreas
frageis['categoria'].value_counts()

63 de 462 linhas são frágeis


categoria
analfabeto/infantil    18
indígena               15
não tem renda          15
mais de 10 SM           7
amarela                 4
preta                   2
1 a 2 SM                1
até 1 SM                1
Name: count, dtype: int64

In [43]:
# Passa na regra do caderno (n de 50 pra cima), mas é frágil (n efetivo menor que 10)
passa_mas_fragil = tabelao[(tabelao['n'] >= 50) & (tabelao['n_efetivo'] < 10)]
print(len(passa_mas_fragil), 'linhas')
passa_mas_fragil.sort_values('n', ascending=False).head(8)

20 linhas


,ano,area,dimensao,categoria,n,resolveram,pct_sem_peso,pct_com_peso,n_efetivo
127,2023,transporte,renda_familiar,1 a 2 SM,443,127,28.7,48.7,9
252,2024,polícia,renda_familiar,até 1 SM,296,91,30.7,60.2,6
281,2024,transporte,raca,preta,210,85,40.5,27.4,6
260,2024,polícia,raca,preta,184,70,38.0,19.3,3
12,2023,documentos,escolaridade,analfabeto/infantil,95,20,21.1,26.5,9
32,2023,saúde,raca,indígena,80,12,15.0,4.6,7
158,2024,documentos,raca,indígena,73,23,31.5,27.4,7
152,2024,documentos,renda_familiar,mais de 10 SM,72,31,43.1,57.0,9
